In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
PPATH = os.getenv("PPATH")
COLOR = os.getenv("COLOR")

print(PPATH)

/Users/daragama/Documents/ProyectosVarios/Datathon2026/


In [7]:
# Cargar bases de datos
b1 = pd.read_csv(PPATH + 'data/hey_clientes.csv')
b2 = pd.read_csv(PPATH + 'data/hey_productos.csv')
b3 = pd.read_csv(PPATH + 'data/hey_transacciones.csv')
#b4 = pd.read_csv('dataset_50k_anonymized.csv')  # input, output, user_id

In [8]:
# ── Agrega Base2 por usuario ──────────────────────────────────────────
# Pivoteamos los productos para tener una fila por usuario
b2_pivot = b2.pivot_table(
    index="user_id",
    columns="tipo_producto",
    values="saldo_actual",
    aggfunc="sum",
    fill_value=0
).reset_index()

b2_extra = b2.groupby("user_id").agg(
    num_productos=("producto_id", "count"),
    utilizacion_media=("utilizacion_pct", "mean"),
    saldo_total=("saldo_actual", "sum"),
).reset_index()

In [9]:
# ── Agrega Base3 por usuario ──────────────────────────────────────────
b3_agg = b3.groupby("user_id").agg(
    total_transacciones=("transaccion_id", "count"),
    monto_total=("monto", "sum"),
    cashback_total=("cashback_generado", "sum"),
    pct_internacional=("es_internacional", "mean"),
    pct_atipico=("patron_uso_atipico", "mean"),
).reset_index()

In [10]:
# ── Merge final ───────────────────────────────────────────────────────
df = (b1
      .merge(b2_pivot,  on="user_id", how="left")
      .merge(b2_extra,  on="user_id", how="left")
      .merge(b3_agg,    on="user_id", how="left"))

df.fillna(0)
print(df.shape)  # (n_usuarios, n_features)


(15025, 41)


In [11]:
print(f"Dataset final: {df.shape}")
print(f"Variables disponibles: {len(df.columns)}")

Dataset final: (15025, 41)
Variables disponibles: 41


# Cargar los clusters de conversación

In [13]:
clusters_input  = pd.read_parquet(PPATH + 'data/data_out/dataset_clusters_in.parquet')  # user_id, cluster
clusters_output = pd.read_parquet(PPATH + 'data/data_out/dataset_clusters_output.parquet') # user_id, cluster

clusters_input.rename(columns={"cluster": "cluster_input"}, inplace=True)
clusters_output.rename(columns={"cluster": "cluster_output"}, inplace=True)

df = (df
      .merge(clusters_input,  on="user_id", how="left")
      .merge(clusters_output, on="user_id", how="left"))

# Preparar features para el modelo de satisfacción

In [14]:
from sklearn.preprocessing import LabelEncoder

# Variables a usar como predictoras
FEATURES = [
    "edad", "ingreso_mensual_mxn", "antiguedad_dias", "score_buro",
    "dias_desde_ultimo_login", "num_productos_activos",
    "es_hey_pro", "nomina_domiciliada", "recibe_remesas", "usa_hey_shop",
    "tiene_seguro", "patron_uso_atipico",
    "num_productos", "utilizacion_media", "saldo_total",
    "total_transacciones", "monto_total", "cashback_total",
    "pct_internacional", "pct_atipico",
    "conv_cluster_x", "conv_cluster_y",
]

TARGET = "satisfaccion_1_10"

# Codificar booleanos
bool_cols = df.select_dtypes(include="bool").columns
df[bool_cols] = df[bool_cols].astype(int)

print(df.columns)

X = df[FEATURES].fillna(0)
y = df[TARGET]

Index(['user_id', 'edad', 'sexo', 'estado', 'ciudad', 'nivel_educativo',
       'ocupacion', 'ingreso_mensual_mxn', 'antiguedad_dias', 'es_hey_pro',
       'nomina_domiciliada', 'canal_apertura', 'score_buro',
       'dias_desde_ultimo_login', 'preferencia_canal', 'satisfaccion_1_10',
       'recibe_remesas', 'usa_hey_shop', 'idioma_preferido', 'tiene_seguro',
       'num_productos_activos', 'patron_uso_atipico', 'credito_auto',
       'credito_nomina', 'credito_personal', 'cuenta_debito',
       'cuenta_negocios', 'inversion_hey', 'seguro_compras', 'seguro_vida',
       'tarjeta_credito_garantizada', 'tarjeta_credito_hey',
       'tarjeta_credito_negocios', 'num_productos', 'utilizacion_media',
       'saldo_total', 'total_transacciones', 'monto_total', 'cashback_total',
       'pct_internacional', 'pct_atipico', 'conv_id_x', 'conv_cluster_x',
       'conv_id_y', 'conv_cluster_y'],
      dtype='str')


In [15]:
print(f"Nulos en satisfaccion_1_10: {y.isna().sum()}")
print(f"Infinitos: {np.isinf(y).sum()}")

mask = y.notna() & np.isfinite(y)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Filas válidas para entrenar: {len(y)}")

Nulos en satisfaccion_1_10: 2804
Infinitos: 0
Filas válidas para entrenar: 52781


# Entrenar modelo de satisfacción con XGBoost

In [16]:
! pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 12.0 MB/s  0:00:00 11.8 MB/s eta 0:00:01

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [18]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = XGBRegressor(n_estimators=300, max_depth=5,
                      learning_rate=0.05, random_state=42)
modelo.fit(X_train, y_train)

mae = mean_absolute_error(y_test, modelo.predict(X_test))
print(f"MAE: {mae:.3f}")  # Error promedio

MAE: 0.807


# Identificar qué variables más afectan la satisfacción

In [19]:
importances = pd.Series(modelo.feature_importances_, index=FEATURES)
top_features = importances.sort_values(ascending=False).head(10)
print(top_features)

utilizacion_media          0.324913
dias_desde_ultimo_login    0.295354
num_productos              0.054288
score_buro                 0.044526
nomina_domiciliada         0.032065
cashback_total             0.029035
total_transacciones        0.027202
monto_total                0.019946
recibe_remesas             0.018049
saldo_total                0.017580
dtype: float32


# Definir las reglas de sugerencia por cluster

In [20]:
# Descripción de cada cluster (la defines explorando los centroides del PCA)
CLUSTER_INPUT_DESC = {
    0: "usuario que pregunta sobre saldos y movimientos",
    1: "usuario que pregunta sobre productos y beneficios",
    2: "usuario que reporta problemas o quejas",
}

CLUSTER_OUTPUT_DESC = {
    0: "respuestas cortas y directas",
    1: "respuestas explicativas con contexto",
    2: "respuestas con opciones y alternativas",
}

# Reglas de sugerencia basadas en variables del modelo
def generar_sugerencia(row, top_features):
    sugerencias = []
    feat_principal = top_features.index[0]

    if feat_principal == "dias_desde_ultimo_login" and row["dias_desde_ultimo_login"] > 30:
        sugerencias.append("Invitar al usuario a explorar su app con recordatorio de funciones nuevas.")

    if feat_principal == "utilizacion_media" and row.get("utilizacion_media", 0) > 0.8:
        sugerencias.append("Ofrecer aumento de límite o productos de crédito complementarios.")

    if feat_principal == "cashback_total" and row.get("cashback_total", 0) == 0:
        sugerencias.append("Informar sobre el programa de cashback que aún no ha aprovechado.")

    if row.get("tiene_seguro", 0) == 0:
        sugerencias.append("Presentar beneficios del seguro de vida o compras.")

    if not sugerencias:
        sugerencias.append("Mantener la conversación con información sobre sus productos actuales.")

    return sugerencias

# Función principal del bot

In [21]:
def bot_personalizado(user_id: str):
    # 1. Obtener perfil del usuario
    row = df[df["user_id"] == user_id]
    #print(row.columns)
    if row.empty:
        return "Usuario no encontrado."
    row = row.iloc[0]

    # 2. Predecir satisfacción actual
    x_user = pd.DataFrame([row[FEATURES]])
    sat_pred = modelo.predict(x_user)[0]

    # 3. Obtener clusters de conversación
    cl_in  = int(row["conv_cluster_x"])
    cl_out = int(row["conv_cluster_y"])

    # 4. Generar sugerencias
    sugerencias = generar_sugerencia(row, top_features)

    # 5. Construir respuesta personalizada
    perfil_conv  = CLUSTER_INPUT_DESC.get(cl_in, "perfil desconocido")
    estilo_resp  = CLUSTER_OUTPUT_DESC.get(cl_out, "estilo estándar")

    respuesta = (
        f"[Bot interno — usuario: {user_id}]\n"
        f"Satisfacción estimada: {sat_pred:.1f}/10\n"
        f"Perfil de conversación: {perfil_conv}\n"
        f"Estilo de respuesta sugerido: {estilo_resp}\n\n"
        f"Sugerencias para mejorar satisfacción:\n"
    )
    for i, s in enumerate(sugerencias, 1):
        respuesta += f"  {i}. {s}\n"

    return respuesta



In [ ]:

print(bot_personalizado("USR-00003"))

[Bot interno — usuario: USR-00003]
Satisfacción estimada: 8.4/10
Perfil de conversación: usuario que reporta problemas o quejas
Estilo de respuesta sugerido: respuestas cortas y directas

Sugerencias para mejorar satisfacción:
  1. Presentar beneficios del seguro de vida o compras.

